In [ ]:
# Orchestrator Worker
# In the orchestrator worker workflow,a central llm breaks down tasks,
# delegates them to worker LLM's and synthesise their results.
# When to use this workflow: 
# This workflow is well suited for complex tasks where you can't 
# predict the subtasks needed(for example-> in coding,number of files 
# that needs to be changed and the nature of change in each file 
# likely depend on the task)
# Whereas it's topographically similar,the key differences from 
# parallelism is flexibility-> subtasks aren't predefined and 
# determined by the orchestrator based on the specific input.

import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model = "openai/gpt-oss-20b")
result = llm.invoke("Hello")
result

In [ ]:
from typing import Annotated,List
import operator 
from typing_extensions import Literal 
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,SystemMessage
from typing_extensions import TypedDict

# Schema for structured output to use in planning
class Section(BaseModel):
    name:str=Field(description="Name for this section of the report")
    description:str=Field(description="Brief overview of main topics and concepts of the section")

class Sections(BaseModel):
    sections:List[Section] = Field(
        description = "Sections of the report"
    )
planner = llm.with_structured_output(Sections)



In [ ]:
## Creating workers dynamically in langgraph 
## LangGraph has the Send API to support orchestrator-worker workflows
## It let you dynamically create worker nodes and send each one to a 
## specific input. Each worker has it's own state and all worker 
## outputs are written to a shared state key that is accessible 
## to the orchestrator graph. This gives the orchestrator access 
## to all worker outputs and allows it to synthesise them to final 
## output. 
from langgraph.types import Send
class State(TypedDict):
    topic:str
    sections:list[Section]
    completed_sections:Annotated[list,operator.add]
    final_report:str

class WorkerState(TypedDict):
    section:Section 
    completed_sections:Annotated[list,operator.add]







In [ ]:
def orchestrator(state:State):
    """Orchestrator that generates a plan for the report."""
    report_sections = planner.invoke([
        SystemMessage(content="Generate a plan for the report"),
        HumanMessage(content=f"What is the report topic: {state['topic']}"),
    ])
    print("Report sections", report_sections)
    return {"sections": report_sections.sections}

In [ ]:
def llm_call(state:WorkerState):
    """Worker writes a section of the report."""
    section = llm.invoke([
        SystemMessage(
            content="Write a report section following the provided name and description. Include no preamble for each section"
        ),
        HumanMessage(
            content=f"Here is the section name: {state['section'].name} and description: {state['section'].description}"
        )
    ])
    return {"completed_sections": [section.content]}

In [ ]:
def assign_workers(state:State):
    """Assign a worker to each section of the plan"""
    return [Send("llm_call",{"section":s}) for s in state["sections"]]


In [ ]:
def synthesizer(state:State):
    """Synthesize full report from sections"""
    completed_sections = state["completed_sections"]
    completed_report_sections = "\n\n....\n".join(completed_sections)
    return {"final_report":completed_report_sections}


In [ ]:
from langgraph.graph import StateGraph,START,END
from IPython.display import Image,display
orchestrator_worker_builder = StateGraph(State)
orchestrator_worker_builder.add_node("orchestrator",orchestrator)
orchestrator_worker_builder.add_node("llm_call",llm_call)
orchestrator_worker_builder.add_node("synthesizer",synthesizer)

orchestrator_worker_builder.add_edge(START,"orchestrator")
orchestrator_worker_builder.add_conditional_edges(
    "orchestrator",assign_workers,["llm_call"]
)
orchestrator_worker_builder.add_edge("llm_call","synthesizer")
orchestrator_worker_builder.add_edge("synthesizer",END)

orchestrator_worker = orchestrator_worker_builder.compile()
display(Image(orchestrator_worker.get_graph().draw_mermaid_png()))


In [ ]:
state = orchestrator_worker.invoke({"topic": "The impact of AI on education"})
print(state["final_report"])